In [0]:
%python
dbutils.widgets.text("caso", "monark", "Caso")
dbutils.widgets.text("versao", "v0.2.0-dev", "Versão")

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

CREATE OR REPLACE VIEW silver.v_normalizado
AS
WITH graphql AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.legacy.id_str') AS id_nativo,
    get_json_object(r.payload,'$.core.user_results.result.core.screen_name') AS autor_handle,
    get_json_object(r.payload,'$.core.user_results.result.rest_id') AS autor_id_nativo,
    to_timestamp(substring(get_json_object(r.payload,'$.legacy.created_at'), 5), 'MMM dd HH:mm:ss Z yyyy') AS created_at,
    get_json_object(r.payload,'$.legacy.full_text') AS texto,
    get_json_object(r.payload,'$.legacy.lang') AS idioma,
    get_json_object(r.payload,'$.legacy.in_reply_to_status_id_str') AS ref_status_id,
    get_json_object(r.payload,'$.legacy.in_reply_to_screen_name') AS ref_handle,
    CAST(get_json_object(r.payload,'$.legacy.is_quote_status') AS BOOLEAN) AS eh_quote,
    CAST(get_json_object(r.payload,'$.legacy.favorite_count') AS INT) AS likes,
    CAST(get_json_object(r.payload,'$.legacy.retweet_count') AS INT) AS retweets,
    CAST(get_json_object(r.payload,'$.legacy.quote_count') AS INT) AS quotes,
    CAST(get_json_object(r.payload,'$.legacy.reply_count') AS INT) AS respostas,
    transform(from_json(get_json_object(r.payload,'$.legacy.entities.user_mentions'), 'array<struct<id_str:string, screen_name:string>>'), m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.screen_name))) AS mencoes,
    transform(from_json(get_json_object(r.payload,'$.legacy.entities.hashtags'), 'array<struct<text:string>>'), h -> lower(h.text)) AS hashtags,
    CAST(NULL AS STRING) AS stance_previa,
    'graphql' AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo a USING (arquivo_id)
  WHERE a.fonte = 'graphql'
),
consolidado AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.id') AS id_nativo,
    get_json_object(r.payload,'$.user') AS autor_handle,
    CAST(NULL AS STRING) AS autor_id_nativo,
    to_timestamp(get_json_object(r.payload,'$.created_at_iso')) AS created_at,
    get_json_object(r.payload,'$.text') AS texto,
    CAST(NULL AS STRING) AS idioma,
    nullif(get_json_object(r.payload,'$.in_reply_to_status_id'),'') AS ref_status_id,
    nullif(get_json_object(r.payload,'$.in_reply_to_user'),'') AS ref_handle,
    CAST(get_json_object(r.payload,'$.is_quote') AS BOOLEAN) AS eh_quote,
    CAST(get_json_object(r.payload,'$.like_count') AS INT) AS likes,
    CAST(get_json_object(r.payload,'$.retweet_count') AS INT) AS retweets,
    CAST(get_json_object(r.payload,'$.quote_count') AS INT) AS quotes,
    CAST(get_json_object(r.payload,'$.reply_count') AS INT) AS respostas,
    transform(from_json(get_json_object(r.payload,'$.mentions'), 'array<struct<id_str:string, username:string>>'), m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.username))) AS mencoes,
    transform(from_json(get_json_object(r.payload,'$.hashtags'), 'array<string>'), h -> lower(h)) AS hashtags,
    nullif(get_json_object(r.payload,'$.stance'),'') AS stance_previa,
    'consolidado' AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo a USING (arquivo_id)
  WHERE a.fonte = 'consolidado'
),
uniao AS (SELECT * FROM graphql UNION ALL SELECT * FROM consolidado)
SELECT *, CASE WHEN ref_status_id IS NOT NULL OR ref_handle IS NOT NULL THEN 'reply' WHEN eh_quote THEN 'quote' ELSE 'original' END AS tipo_ref FROM uniao;

CREATE OR REPLACE VIEW silver.v_deduplicado
AS
SELECT caso_slug, id_nativo,
  min_by(autor_handle, linha) AS autor_handle,
  min_by(autor_id_nativo, linha) AS autor_id_nativo,
  min_by(created_at, linha) AS created_at,
  min_by(texto, linha) AS texto,
  min_by(idioma, linha) AS idioma,
  min_by(ref_status_id, linha) AS ref_status_id,
  min_by(ref_handle, linha) AS ref_handle,
  min_by(tipo_ref, linha) AS tipo_ref,
  min_by(mencoes, linha) AS mencoes,
  min_by(hashtags, linha) AS hashtags,
  min_by(stance_previa, linha) AS stance_previa,
  min_by(fonte, linha) AS fonte,
  max(likes) AS likes,
  max(retweets) AS retweets,
  max(quotes) AS quotes,
  max(respostas) AS respostas,
  count(*) AS capturas
FROM silver.v_normalizado
GROUP BY caso_slug, id_nativo;

SELECT 'Views criadas com sucesso' AS status;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

-- =====================================================================
-- =====================================================================

CREATE OR REPLACE VIEW silver.v_normalizado
COMMENT 'Une as duas fontes de ingestão num único formato. É o normalizador do pipeline.'
AS
WITH graphql AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.legacy.id_str')                                AS id_nativo,
    get_json_object(r.payload,'$.core.user_results.result.core.screen_name')    AS autor_handle,
    get_json_object(r.payload,'$.core.user_results.result.rest_id')             AS autor_id_nativo,
    to_timestamp(substring(get_json_object(r.payload,'$.legacy.created_at'), 5),
                 'MMM dd HH:mm:ss Z yyyy')                                  AS created_at,
    get_json_object(r.payload,'$.legacy.full_text')                             AS texto,
    get_json_object(r.payload,'$.legacy.lang')                                  AS idioma,
    get_json_object(r.payload,'$.legacy.in_reply_to_status_id_str')             AS ref_status_id,
    get_json_object(r.payload,'$.legacy.in_reply_to_screen_name')               AS ref_handle,
    CAST(get_json_object(r.payload,'$.legacy.is_quote_status') AS BOOLEAN)      AS eh_quote,
    CAST(get_json_object(r.payload,'$.legacy.favorite_count') AS INT)           AS likes,
    CAST(get_json_object(r.payload,'$.legacy.retweet_count')  AS INT)           AS retweets,
    CAST(get_json_object(r.payload,'$.legacy.quote_count')    AS INT)           AS quotes,
    CAST(get_json_object(r.payload,'$.legacy.reply_count')    AS INT)           AS respostas,
    transform(
      from_json(get_json_object(r.payload,'$.legacy.entities.user_mentions'),
                'array<struct<id_str:string, screen_name:string>>'),
      m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.screen_name))
    )                                                                           AS mencoes,
    transform(
      from_json(get_json_object(r.payload,'$.legacy.entities.hashtags'),
                'array<struct<text:string>>'),
      h -> lower(h.text)
    )                                                                           AS hashtags,
    CAST(NULL AS STRING)                                                        AS stance_previa,
    'graphql'                                                                   AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo  a USING (arquivo_id)
  WHERE a.fonte = 'graphql'
),
consolidado AS (
  SELECT
    a.arquivo_id, a.caso_slug, r.linha,
    get_json_object(r.payload,'$.id')                                           AS id_nativo,
    get_json_object(r.payload,'$.user')                                         AS autor_handle,
    CAST(NULL AS STRING)                                                        AS autor_id_nativo,
    to_timestamp(get_json_object(r.payload,'$.created_at_iso'))                 AS created_at,
    get_json_object(r.payload,'$.text')                                         AS texto,
    CAST(NULL AS STRING)                                                        AS idioma,
    nullif(get_json_object(r.payload,'$.in_reply_to_status_id'),'')             AS ref_status_id,
    nullif(get_json_object(r.payload,'$.in_reply_to_user'),'')                  AS ref_handle,
    CAST(get_json_object(r.payload,'$.is_quote') AS BOOLEAN)                    AS eh_quote,
    CAST(get_json_object(r.payload,'$.like_count')    AS INT)                   AS likes,
    CAST(get_json_object(r.payload,'$.retweet_count') AS INT)                   AS retweets,
    CAST(get_json_object(r.payload,'$.quote_count')   AS INT)                   AS quotes,
    CAST(get_json_object(r.payload,'$.reply_count')   AS INT)                   AS respostas,
    transform(
      from_json(get_json_object(r.payload,'$.mentions'),
                'array<struct<id_str:string, username:string>>'),
      m -> named_struct('id_nativo', m.id_str, 'handle', lower(m.username))
    )                                                                           AS mencoes,
    transform(
      from_json(get_json_object(r.payload,'$.hashtags'), 'array<string>'),
      h -> lower(h)
    )                                                                           AS hashtags,
    nullif(get_json_object(r.payload,'$.stance'),'')                            AS stance_previa,
    'consolidado'                                                               AS fonte
  FROM bronze.registro r
  JOIN bronze.arquivo  a USING (arquivo_id)
  WHERE a.fonte = 'consolidado'
),
uniao AS (SELECT * FROM graphql UNION ALL SELECT * FROM consolidado)
SELECT
  *,
  CASE
    WHEN ref_status_id IS NOT NULL OR ref_handle IS NOT NULL THEN 'reply'
    WHEN eh_quote                                            THEN 'quote'
    ELSE 'original'
  END AS tipo_ref
FROM uniao;

-- =====================================================================
-- =====================================================================

CREATE OR REPLACE VIEW silver.v_deduplicado
COMMENT 'Uma linha por postagem. Contadores = máximo observado entre as capturas do mesmo id.'
AS
SELECT
  caso_slug, id_nativo,
  min_by(autor_handle,    linha) AS autor_handle,
  min_by(autor_id_nativo, linha) AS autor_id_nativo,
  min_by(created_at,      linha) AS created_at,
  min_by(texto,           linha) AS texto,
  min_by(idioma,          linha) AS idioma,
  min_by(ref_status_id,   linha) AS ref_status_id,
  min_by(ref_handle,      linha) AS ref_handle,
  min_by(tipo_ref,        linha) AS tipo_ref,
  min_by(mencoes,         linha) AS mencoes,
  min_by(hashtags,        linha) AS hashtags,
  min_by(stance_previa,   linha) AS stance_previa,
  min_by(fonte,           linha) AS fonte,
  max(likes)     AS likes,
  max(retweets)  AS retweets,
  max(quotes)    AS quotes,
  max(respostas) AS respostas,
  count(*)       AS capturas
FROM silver.v_normalizado
GROUP BY caso_slug, id_nativo;



In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

CREATE OR REPLACE VIEW silver.v_promovivel
AS
SELECT * FROM silver.v_deduplicado
WHERE autor_handle IS NOT NULL AND trim(autor_handle) <> '';

MERGE INTO silver.conta AS alvo
USING (
  SELECT handle, max(id_nativo) AS id_nativo FROM (
    SELECT lower(autor_handle) AS handle, autor_id_nativo AS id_nativo FROM silver.v_promovivel WHERE caso_slug = :caso
    UNION ALL
    SELECT m.handle, m.id_nativo FROM silver.v_promovivel LATERAL VIEW explode(mencoes) t AS m WHERE caso_slug = :caso AND m.handle IS NOT NULL
    UNION ALL
    SELECT lower(ref_handle), NULL FROM silver.v_promovivel WHERE caso_slug = :caso AND ref_handle IS NOT NULL
  ) GROUP BY handle
) AS origem
ON alvo.plataforma = 'x' AND alvo.handle = origem.handle
WHEN MATCHED AND alvo.id_nativo IS NULL AND origem.id_nativo IS NOT NULL THEN UPDATE SET alvo.id_nativo = origem.id_nativo
WHEN NOT MATCHED THEN INSERT (plataforma, handle, id_nativo, criado_em) VALUES ('x', origem.handle, origem.id_nativo, current_timestamp());

SELECT COUNT(*) AS contas FROM silver.conta;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

MERGE INTO silver.postagem AS alvo
USING (
  SELECT d.id_nativo, d.caso_slug, c.conta_id AS autor_conta_id, d.created_at, d.texto, d.idioma, d.tipo_ref, d.ref_status_id,
         cr.conta_id AS ref_conta_id, d.fonte
  FROM silver.v_promovivel d
  JOIN silver.conta c ON c.plataforma='x' AND c.handle = lower(d.autor_handle)
  LEFT JOIN silver.conta cr ON cr.plataforma='x' AND cr.handle = lower(d.ref_handle)
  WHERE d.caso_slug = :caso
) AS origem
ON alvo.plataforma = 'x' AND alvo.id_nativo = origem.id_nativo
WHEN NOT MATCHED THEN INSERT (plataforma, id_nativo, caso_slug, autor_conta_id, created_at, texto, idioma, tipo_ref, ref_id_nativo, ref_conta_id, situacao, fonte, carregada_em)
VALUES ('x', origem.id_nativo, origem.caso_slug, origem.autor_conta_id, origem.created_at, origem.texto, origem.idioma, origem.tipo_ref, origem.ref_status_id, origem.ref_conta_id, 'ativa', origem.fonte, current_timestamp());

SELECT COUNT(*) AS postagens FROM silver.postagem;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

INSERT INTO silver.captura (postagem_id, likes, retweets, quotes, respostas, capturas, capturado_em)
SELECT 
  p.postagem_id, 
  d.likes, 
  d.retweets, 
  d.quotes, 
  d.respostas, 
  d.capturas, 
  current_timestamp()
FROM silver.v_promovivel d
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = d.id_nativo
WHERE d.caso_slug = :caso;

SELECT COUNT(*) AS total_capturas FROM silver.captura;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

INSERT INTO silver.mencao (postagem_id, conta_id)
SELECT DISTINCT
  p.postagem_id,
  c.conta_id
FROM (
  SELECT d.id_nativo, mencao.handle as mention_handle
  FROM silver.v_promovivel d
  LATERAL VIEW explode(d.mencoes) tbl AS mencao
  WHERE d.caso_slug = :caso AND mencao.handle IS NOT NULL
) exploded
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = exploded.id_nativo
JOIN silver.conta c 
  ON c.plataforma='x' AND c.handle = exploded.mention_handle;

SELECT COUNT(*) AS total_mencoes FROM silver.mencao;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

INSERT INTO silver.postagem_hashtag (postagem_id, hashtag)
SELECT DISTINCT
  p.postagem_id,
  exploded.hashtag
FROM (
  SELECT d.id_nativo, hashtag
  FROM silver.v_promovivel d
  LATERAL VIEW explode(d.hashtags) tbl AS hashtag
  WHERE d.caso_slug = :caso AND hashtag IS NOT NULL
) exploded
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = exploded.id_nativo;

SELECT COUNT(*) AS total_hashtags FROM silver.postagem_hashtag;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

INSERT INTO silver.classificacao (postagem_id, esquema, rotulo, modelo, versao, confianca, classificado_em)
SELECT 
  p.postagem_id,
  'stance',
  d.stance_previa,
  'bertimbau-stance',
  'previa-sem-confianca',
  CAST(NULL AS DOUBLE),
  current_timestamp()
FROM silver.v_promovivel d
JOIN silver.postagem p 
  ON p.plataforma='x' AND p.id_nativo = d.id_nativo
WHERE d.caso_slug = :caso 
  AND d.stance_previa IS NOT NULL;

SELECT COUNT(*) AS total_classificacoes FROM silver.classificacao;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT 'conta' AS tabela, COUNT(*) AS linhas FROM silver.conta
UNION ALL SELECT 'postagem', COUNT(*) FROM silver.postagem
UNION ALL SELECT 'captura', COUNT(*) FROM silver.captura
UNION ALL SELECT 'mencao', COUNT(*) FROM silver.mencao
UNION ALL SELECT 'postagem_hashtag', COUNT(*) FROM silver.postagem_hashtag
UNION ALL SELECT 'classificacao', COUNT(*) FROM silver.classificacao
ORDER BY tabela;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT postagem_id, COUNT(*) AS n
FROM silver.captura
GROUP BY postagem_id
HAVING COUNT(*) > 1
LIMIT 10;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

CREATE TABLE IF NOT EXISTS silver.captura_quarentena AS
SELECT c.*
FROM silver.captura c
WHERE c.captura_id NOT IN (
  SELECT MIN(captura_id) FROM silver.captura GROUP BY postagem_id
);

DELETE FROM silver.captura
WHERE captura_id NOT IN (
  SELECT MIN(captura_id) FROM silver.captura GROUP BY postagem_id
);

SELECT COUNT(*) AS capturas_restantes FROM silver.captura;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

SELECT p.caso_slug,
       COUNT(DISTINCT p.postagem_id)  AS postagens,
       COUNT(c.captura_id)            AS capturas,
       COUNT(DISTINCT m.mencao_id)    AS mencoes,
       COUNT(DISTINCT h.hashtag)      AS hashtags_distintas,
       COUNT(DISTINCT k.classificacao_id) AS classificacoes
FROM silver.postagem p
LEFT JOIN silver.captura c          ON c.postagem_id = p.postagem_id
LEFT JOIN silver.mencao m           ON m.postagem_id = p.postagem_id
LEFT JOIN silver.postagem_hashtag h ON h.postagem_id = p.postagem_id
LEFT JOIN silver.classificacao k    ON k.postagem_id = p.postagem_id
GROUP BY p.caso_slug
ORDER BY p.caso_slug;

In [0]:
USE CATALOG scapegoat;
USE SCHEMA silver;

WITH
pst AS (SELECT caso_slug, COUNT(*) AS postagens FROM silver.postagem GROUP BY caso_slug),
cap AS (SELECT p.caso_slug, COUNT(*) AS capturas
        FROM silver.captura c JOIN silver.postagem p ON p.postagem_id = c.postagem_id
        GROUP BY p.caso_slug),
men AS (SELECT p.caso_slug, COUNT(*) AS mencoes
        FROM silver.mencao m JOIN silver.postagem p ON p.postagem_id = m.postagem_id
        GROUP BY p.caso_slug),
hsh AS (SELECT p.caso_slug, COUNT(*) AS pares_postagem_hashtag
        FROM silver.postagem_hashtag h JOIN silver.postagem p ON p.postagem_id = h.postagem_id
        GROUP BY p.caso_slug)
SELECT pst.caso_slug, pst.postagens, cap.capturas, men.mencoes, hsh.pares_postagem_hashtag
FROM pst
LEFT JOIN cap ON cap.caso_slug = pst.caso_slug
LEFT JOIN men ON men.caso_slug = pst.caso_slug
LEFT JOIN hsh ON hsh.caso_slug = pst.caso_slug
ORDER BY pst.caso_slug;